# 01 - Manutencao Preventiva de Tornos
> Analise de vibracoes, deteccao de anomalias e series temporais

**Modulo:** `13_enfitec_projetos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('numpy e matplotlib importados!')

## Gerando Dados de Vibracao Simulados

Vamos simular dados de vibracao de um torno mecanico em tres cenarios:
- **Normal:** operacao sem falhas
- **Desbalanceamento:** frequencia fundamental elevada
- **Defeito no rolamento:** picos de alta frequencia periodicos

In [ ]:
# Parametros de simulacao
fs = 1000          # Taxa de amostragem (Hz)
t = np.linspace(0, 2, 2 * fs)  # 2 segundos
rpm = 1800         # Rotacao do torno
f_rot = rpm / 60   # Frequencia de rotacao (Hz)

# --- Sinal Normal ---
normal = (0.5 * np.sin(2 * np.pi * f_rot * t) +
          0.2 * np.sin(2 * np.pi * 2 * f_rot * t) +
          0.05 * np.random.randn(len(t)))

# --- Desbalanceamento ---
desbalanceamento = (1.8 * np.sin(2 * np.pi * f_rot * t) +
                    0.3 * np.sin(2 * np.pi * 2 * f_rot * t) +
                    0.1 * np.random.randn(len(t)))

# --- Defeito no Rolamento ---
f_bpfo = 4.2 * f_rot  # Frequencia de defeito externo
rolamento = (0.5 * np.sin(2 * np.pi * f_rot * t) +
             0.2 * np.sin(2 * np.pi * 2 * f_rot * t) +
             0.7 * np.sin(2 * np.pi * f_bpfo * t) +
             0.4 * np.sin(2 * np.pi * 2 * f_bpfo * t) +
             0.08 * np.random.randn(len(t)))

print(f'Frequencia de rotacao: {f_rot} Hz')
print(f'Frequencia BPFO: {f_bpfo:.1f} Hz')
print(f'Amostras geradas: {len(t)}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t[:500], normal[:500], 'g', linewidth=0.8)
axes[0].set_title('Vibracao Normal')
axes[0].set_ylabel('Amplitude (mm/s)')

axes[1].plot(t[:500], desbalanceamento[:500], 'orange', linewidth=0.8)
axes[1].set_title('Desbalanceamento')
axes[1].set_ylabel('Amplitude (mm/s)')

axes[2].plot(t[:500], rolamento[:500], 'r', linewidth=0.8)
axes[2].set_title('Defeito no Rolamento')
axes[2].set_ylabel('Amplitude (mm/s)')
axes[2].set_xlabel('Tempo (s)')

plt.tight_layout()
plt.savefig('vibracoes_simuladas.png', dpi=100, bbox_inches='tight')
plt.show()
print('Graficos de vibracao gerados!')

## Analise FFT de Vibracoes

Vamos pedir ao Claude para implementar analise FFT que identifique
frequencias caracteristicas de cada modo de falha.

In [ ]:
prompt_fft = """Implemente em Python uma funcao de analise FFT para sinais de vibracao
de maquinas rotativas. A funcao deve:

1. Receber o sinal temporal e a taxa de amostragem
2. Aplicar janela de Hanning antes da FFT
3. Calcular o espectro de amplitude unilateral
4. Identificar os picos dominantes (frequencia e amplitude)
5. Classificar o tipo de falha com base nos picos:
   - Desbalanceamento: pico dominante em 1x RPM
   - Desalinhamento: pico dominante em 2x RPM
   - Defeito no rolamento: picos em frequencias BPFO (tipicamente 4-6x RPM)
   - Engrenagem danificada: picos em frequencias de engrenamento

Retorne codigo Python completo com a funcao e exemplo de uso.
Use numpy e scipy. Frequencia de rotacao = 30 Hz."""

resp_fft = ask(prompt_fft, model=SONNET, max_tokens=2048)
print(resp_fft)

In [ ]:
from scipy.signal import find_peaks

def analise_fft(sinal, fs, f_rot, n_picos=5):
    """Analise FFT com janela de Hanning e deteccao de picos."""
    N = len(sinal)
    janela = np.hanning(N)
    sinal_jan = sinal * janela

    fft_vals = np.fft.rfft(sinal_jan)
    freqs = np.fft.rfftfreq(N, 1.0 / fs)
    amplitude = 2.0 * np.abs(fft_vals) / N

    picos_idx, props = find_peaks(amplitude, height=0.05, distance=5)
    picos_ord = picos_idx[np.argsort(props['peak_heights'])[::-1]]
    top_picos = picos_ord[:n_picos]

    return freqs, amplitude, top_picos

# Aplicar FFT nos tres sinais
sinais = {'Normal': normal, 'Desbalanceamento': desbalanceamento, 'Rolamento': rolamento}
resultados_fft = {}

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
cores = {'Normal': 'g', 'Desbalanceamento': 'orange', 'Rolamento': 'r'}

for i, (nome, sinal) in enumerate(sinais.items()):
    freqs, amp, picos = analise_fft(sinal, fs, f_rot)
    resultados_fft[nome] = (freqs, amp, picos)

    axes[i].plot(freqs[:250], amp[:250], cores[nome], linewidth=0.8)
    axes[i].plot(freqs[picos], amp[picos], 'kx', markersize=10)
    axes[i].set_title(f'Espectro FFT - {nome}')
    axes[i].set_ylabel('Amplitude')

    for p in picos[:3]:
        axes[i].annotate(f'{freqs[p]:.1f} Hz',
                         xy=(freqs[p], amp[p]),
                         fontsize=8, ha='center',
                         xytext=(0, 10),
                         textcoords='offset points')

axes[2].set_xlabel('Frequencia (Hz)')
plt.tight_layout()
plt.savefig('analise_fft.png', dpi=100, bbox_inches='tight')
plt.show()
print('Analise FFT concluida!')

## Deteccao de Anomalias

Vamos pedir ao Claude para implementar algoritmos de deteccao de anomalias
baseados em z-score e media movel.

In [ ]:
prompt_anomalia = """Implemente em Python dois metodos de deteccao de anomalias
para series temporais de vibracao de maquinas industriais:

1. Z-Score: calcule o z-score de cada janela de dados e sinalize
   pontos com |z| > threshold como anomalias.

2. Media Movel com Limiar: calcule a media movel e o desvio padrao
   movel. Pontos fora de media +/- k*desvio sao anomalias.

Cada funcao deve receber o sinal e retornar:
- indices das anomalias
- scores de severidade
- limiar usado

Use numpy. Retorne codigo completo e funcional."""

resp_anomalia = ask(prompt_anomalia, model=SONNET, max_tokens=2048)
print(resp_anomalia)

In [ ]:
def detectar_zscore(sinal, janela=100, threshold=3.0):
    """Deteccao de anomalias via z-score por janela deslizante."""
    n = len(sinal)
    z_scores = np.zeros(n)
    anomalias = []

    for i in range(janela, n):
        trecho = sinal[i - janela:i]
        media = np.mean(trecho)
        desvio = np.std(trecho)
        if desvio > 0:
            z_scores[i] = (sinal[i] - media) / desvio
        if abs(z_scores[i]) > threshold:
            anomalias.append(i)

    return np.array(anomalias), z_scores, threshold


def detectar_media_movel(sinal, janela=50, k=2.5):
    """Deteccao de anomalias via media movel + desvio padrao movel."""
    n = len(sinal)
    media_mov = np.convolve(sinal, np.ones(janela) / janela, mode='same')
    desvio_mov = np.array([
        np.std(sinal[max(0, i - janela // 2):i + janela // 2])
        for i in range(n)
    ])

    lim_sup = media_mov + k * desvio_mov
    lim_inf = media_mov - k * desvio_mov
    anomalias = np.where((sinal > lim_sup) | (sinal < lim_inf))[0]
    severidade = np.abs(sinal - media_mov) / (desvio_mov + 1e-8)

    return anomalias, severidade, (lim_inf, lim_sup, media_mov)

print('Funcoes de deteccao de anomalias definidas!')

In [ ]:
# Criar sinal com anomalias injetadas
sinal_teste = normal.copy()
# Injetar spikes
sinal_teste[400] = 5.0
sinal_teste[800] = -4.5
sinal_teste[1200] = 6.0
# Injetar trecho de alta amplitude
sinal_teste[1500:1550] = sinal_teste[1500:1550] * 4

# Aplicar ambos os metodos
anom_z, z_scores, thr_z = detectar_zscore(sinal_teste)
anom_m, sev_m, (li, ls, mm) = detectar_media_movel(sinal_teste)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(t, sinal_teste, 'b', linewidth=0.5, label='Sinal')
if len(anom_z) > 0:
    axes[0].scatter(t[anom_z], sinal_teste[anom_z], c='r', s=30, zorder=5, label='Anomalias')
axes[0].set_title('Sinal com Anomalias Injetadas')
axes[0].legend()

axes[1].plot(t, z_scores, 'purple', linewidth=0.5)
axes[1].axhline(y=thr_z, color='r', linestyle='--', label=f'Limiar = {thr_z}')
axes[1].axhline(y=-thr_z, color='r', linestyle='--')
axes[1].set_title('Z-Score')
axes[1].legend()

axes[2].plot(t, sinal_teste, 'b', linewidth=0.5, label='Sinal')
axes[2].plot(t, mm, 'g', linewidth=1, label='Media Movel')
axes[2].fill_between(t, li, ls, alpha=0.2, color='green', label='Faixa Normal')
if len(anom_m) > 0:
    axes[2].scatter(t[anom_m], sinal_teste[anom_m], c='r', s=10, zorder=5, label='Anomalias')
axes[2].set_title('Media Movel com Limiar')
axes[2].set_xlabel('Tempo (s)')
axes[2].legend()

plt.tight_layout()
plt.savefig('deteccao_anomalias.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Z-Score: {len(anom_z)} anomalias detectadas')
print(f'Media Movel: {len(anom_m)} anomalias detectadas')

## Sistema de Alertas

Vamos pedir ao Claude para projetar a logica de um sistema de monitoramento
com niveis de severidade.

In [ ]:
prompt_alertas = """Projete a logica de um sistema de monitoramento de vibracoes
para tornos mecanicos com os seguintes niveis de severidade:

- NORMAL (verde): vibracoes dentro da faixa esperada
- ATENCAO (amarelo): vibracoes levemente elevadas, agendar inspecao
- ALERTA (laranja): vibracoes significativamente elevadas, inspecionar em breve
- CRITICO (vermelho): vibracoes criticas, parar maquina imediatamente

Para cada nivel, defina:
1. Limiares baseados em RMS e pico de vibracao (em mm/s)
2. Acoes recomendadas
3. Tempo maximo de resposta
4. Quem notificar

Baseie-se na norma ISO 10816 para maquinas Classe I/II.
Retorne como codigo Python (classe ou dicionario estruturado)."""

resp_alertas = ask(prompt_alertas, model=SONNET, max_tokens=2048)
print(resp_alertas)

In [ ]:
class SistemaAlertas:
    """Sistema de alertas baseado na ISO 10816 para maquinas Classe II."""

    NIVEIS = {
        'NORMAL':  {'rms_max': 1.8, 'pico_max': 3.5, 'cor': 'green',
                    'acao': 'Operacao normal. Continuar monitoramento de rotina.',
                    'tempo_resposta': 'Proximo ciclo de manutencao',
                    'notificar': ['Sistema automatico']},
        'ATENCAO': {'rms_max': 4.5, 'pico_max': 7.0, 'cor': 'yellow',
                    'acao': 'Agendar inspecao. Aumentar frequencia de monitoramento.',
                    'tempo_resposta': '7 dias',
                    'notificar': ['Tecnico de manutencao']},
        'ALERTA':  {'rms_max': 11.2, 'pico_max': 18.0, 'cor': 'orange',
                    'acao': 'Inspecionar em breve. Preparar pecas de reposicao.',
                    'tempo_resposta': '24 horas',
                    'notificar': ['Tecnico', 'Supervisor', 'Engenheiro']},
        'CRITICO': {'rms_max': float('inf'), 'pico_max': float('inf'), 'cor': 'red',
                    'acao': 'PARAR MAQUINA IMEDIATAMENTE. Risco de dano catastrofico.',
                    'tempo_resposta': 'Imediato',
                    'notificar': ['Todos', 'Gerente de planta', 'Seguranca']}
    }

    def classificar(self, sinal):
        """Classifica o nivel de severidade do sinal."""
        rms = np.sqrt(np.mean(sinal ** 2))
        pico = np.max(np.abs(sinal))

        for nivel, params in self.NIVEIS.items():
            if rms <= params['rms_max'] and pico <= params['pico_max']:
                return {
                    'nivel': nivel,
                    'rms': rms,
                    'pico': pico,
                    'cor': params['cor'],
                    'acao': params['acao'],
                    'tempo_resposta': params['tempo_resposta'],
                    'notificar': params['notificar']
                }
        return {'nivel': 'CRITICO', 'rms': rms, 'pico': pico,
                'cor': 'red', 'acao': self.NIVEIS['CRITICO']['acao'],
                'tempo_resposta': 'Imediato',
                'notificar': self.NIVEIS['CRITICO']['notificar']}

# Testar com os sinais simulados
sistema = SistemaAlertas()

for nome, sinal in sinais.items():
    resultado = sistema.classificar(sinal)
    print(f'=== {nome} ===')
    print(f'  Nivel: {resultado["nivel"]}')
    print(f'  RMS: {resultado["rms"]:.3f} mm/s')
    print(f'  Pico: {resultado["pico"]:.3f} mm/s')
    print(f'  Acao: {resultado["acao"]}')
    print()

## Dashboard de Monitoramento

Vamos pedir ao Claude para gerar o codigo de um dashboard completo
usando matplotlib subplots.

In [ ]:
prompt_dashboard = """Gere codigo Python usando matplotlib para criar um dashboard
de monitoramento de vibracao de uma maquina rotativa com 4 subplots:

1. Serie temporal do sinal de vibracao (ultimos 2 segundos)
2. Espectro FFT com picos identificados
3. Historico de RMS com limiares de alerta (normal/atencao/alerta/critico)
4. Indicador de status atual (semaforo com nivel de severidade)

Use cores consistentes: verde=normal, amarelo=atencao, laranja=alerta,
vermelho=critico. Inclua titulo, legendas e anotacoes.
Retorne codigo completo e funcional."""

resp_dashboard = ask(prompt_dashboard, model=SONNET, max_tokens=2048)
print(resp_dashboard)

In [ ]:
def criar_dashboard(sinal, fs, f_rot, nome_maquina='Torno CNC #01'):
    """Cria dashboard de monitoramento com 4 paineis."""
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f'Dashboard de Monitoramento - {nome_maquina}',
                 fontsize=14, fontweight='bold')

    # 1. Serie Temporal
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.plot(t[:500], sinal[:500], 'b', linewidth=0.6)
    ax1.set_title('Sinal de Vibracao (Tempo)')
    ax1.set_xlabel('Tempo (s)')
    ax1.set_ylabel('Amplitude (mm/s)')
    ax1.grid(True, alpha=0.3)

    # 2. Espectro FFT
    ax2 = fig.add_subplot(2, 2, 2)
    freqs, amp, picos = analise_fft(sinal, fs, f_rot)
    ax2.plot(freqs[:250], amp[:250], 'darkblue', linewidth=0.8)
    ax2.plot(freqs[picos], amp[picos], 'rv', markersize=8)
    for p in picos[:3]:
        ax2.annotate(f'{freqs[p]:.0f} Hz', xy=(freqs[p], amp[p]),
                     fontsize=7, ha='center', xytext=(0, 8),
                     textcoords='offset points')
    ax2.set_title('Espectro de Frequencia (FFT)')
    ax2.set_xlabel('Frequencia (Hz)')
    ax2.set_ylabel('Amplitude')
    ax2.grid(True, alpha=0.3)

    # 3. Historico RMS (simulado)
    ax3 = fig.add_subplot(2, 2, 3)
    n_janelas = 40
    tam_janela = len(sinal) // n_janelas
    rms_hist = [np.sqrt(np.mean(sinal[i*tam_janela:(i+1)*tam_janela]**2))
                for i in range(n_janelas)]
    tempo_hist = np.arange(n_janelas) * (tam_janela / fs)

    ax3.plot(tempo_hist, rms_hist, 'b-o', markersize=3, linewidth=1.2)
    ax3.axhline(y=1.8, color='green', linestyle='--', alpha=0.7, label='Normal < 1.8')
    ax3.axhline(y=4.5, color='orange', linestyle='--', alpha=0.7, label='Atencao < 4.5')
    ax3.axhline(y=11.2, color='red', linestyle='--', alpha=0.7, label='Critico < 11.2')
    ax3.fill_between(tempo_hist, 0, 1.8, alpha=0.1, color='green')
    ax3.fill_between(tempo_hist, 1.8, 4.5, alpha=0.1, color='yellow')
    ax3.fill_between(tempo_hist, 4.5, 11.2, alpha=0.1, color='orange')
    ax3.set_title('Historico RMS')
    ax3.set_xlabel('Tempo (s)')
    ax3.set_ylabel('RMS (mm/s)')
    ax3.legend(fontsize=7)
    ax3.grid(True, alpha=0.3)

    # 4. Indicador de Status
    ax4 = fig.add_subplot(2, 2, 4)
    ax4.set_xlim(0, 10)
    ax4.set_ylim(0, 10)
    ax4.set_aspect('equal')
    ax4.axis('off')

    resultado = sistema.classificar(sinal)
    cor = resultado['cor']

    circle = plt.Circle((5, 6), 2.5, color=cor, ec='black', linewidth=2)
    ax4.add_patch(circle)
    ax4.text(5, 6, resultado['nivel'], ha='center', va='center',
             fontsize=16, fontweight='bold',
             color='white' if cor in ['red', 'green'] else 'black')
    ax4.text(5, 2.5, f'RMS: {resultado["rms"]:.2f} mm/s',
             ha='center', fontsize=10)
    ax4.text(5, 1.5, f'Pico: {resultado["pico"]:.2f} mm/s',
             ha='center', fontsize=10)
    ax4.text(5, 0.5, resultado['acao'][:50],
             ha='center', fontsize=8, style='italic')
    ax4.set_title('Status Atual')

    plt.tight_layout()
    return fig

# Dashboard para cada sinal
for nome, sinal in sinais.items():
    fig = criar_dashboard(sinal, fs, f_rot, f'Torno CNC #01 - {nome}')
    plt.savefig(f'dashboard_{nome.lower()}.png', dpi=100, bbox_inches='tight')
    plt.show()

print('Dashboards gerados com sucesso!')

## Exercicios

1. **FFT com Resolucao Variavel:** Modifique a funcao `analise_fft` para aceitar
   diferentes tamanhos de janela (256, 512, 1024, 2048). Compare a resolucao em
   frequencia vs. resolucao temporal.

2. **Novos Modos de Falha:** Adicione sinais simulados para engrenagem danificada
   (picos em multiplos da frequencia de engrenamento) e folga mecanica
   (harmonicos sub-sincronos). Teste a deteccao.

3. **Anomalia em Tempo Real:** Adapte a funcao `detectar_media_movel` para
   processar dados em streaming (amostra por amostra) em vez de batch.

4. **Machine Learning:** Peca ao Claude para implementar um classificador
   (SVM ou Random Forest) que diferencie automaticamente entre os modos de falha
   usando features extraidas da FFT.

5. **Dashboard Interativo:** Use Plotly em vez de matplotlib para criar um
   dashboard com zoom, tooltips e atualizacao em tempo real.

## Proximos Passos

- Integrar com sensores reais via protocolo Modbus ou OPC-UA
- Implementar banco de dados de historico (InfluxDB ou TimescaleDB)
- Adicionar modelos preditivos (LSTM, autoencoders) para prognostico
- Criar API REST para integracao com sistemas MES/ERP
- Implementar alertas via e-mail, SMS e push notification
- Desenvolver app mobile para monitoramento remoto

---

*Notebook criado com auxilio do Claude (Anthropic) para o modulo 13 - Enfitec Projetos.*